> 请点击获取[课程 PPT 内容](https://www.canva.cn/design/DAG37LTmMSo/UKy_0jbjl1CSkwK5-iYddA/view?utm_content=DAG37LTmMSo&utm_campaign=designshare&utm_medium=link2&utm_source=uniquelinks&utlId=hf9b020d53f)。


# 1. 环境配置

## 1.1 python 环境准备

In [ ]:
! pip install openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulsoup4==4.14.3 langchain_chroma==1.1.0

## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

# 2. RAG Agent 应用开发

## 2.1 简介

在第四章里我们已经学习过如何创建并使用向量数据库（RAG），那假如我们希望从向量数据库中获取信息并应用到智能体中，我们就需要先完成第一步制作数据库后才能实现。

在完成向量数据库的创建后，我们就可以将检索的过程进行封装，然后输入就是用户提出或智能体润色后的问题，输出就是搜索到的片段信息。

当我们基于前面的代码创建出了一个向量数据库后：

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size = 1500,
  chunk_overlap = 150)
splits = text_splitter.split_documents(docs)

from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os
embeddings = DashScopeEmbeddings(
  dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'), 
  model="text-embedding-v1")
vectordb = Chroma.from_documents(documents=splits,
  embedding=embeddings,
  persist_directory='./chroma')

然后我们就可以通过 Chroma 加载向量数据库：

In [ ]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os

embeddings = DashScopeEmbeddings(
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"), model="text-embedding-v1"
)
vectordb = Chroma(
    embedding_function=embeddings,
    persist_directory="./chroma"  # 必须与创建数据库的路径一致
)

然后将提问→检索向量数据库→回复相关片段这一流程改造成工具来进行使用：

In [ ]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vectordb.similarity_search(query, k=1)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

这里设置的 response_format 表示不仅把拼接后的 serialized 注入到 ToolMessage 的 content 里。同时也会把 retrieved_docs 单独保存到 ToolMessage 的 artifact 字段中。

然后我们需要将大模型、工具、提示词统一配置到 create_agent 中：

In [ ]:
from langchain_community.chat_models import ChatTongyi
from langchain.agents import create_agent

llm = ChatTongyi(model="qwen-max") 

tools = [retrieve_context]

prompt = (
    "You have access to a tool that retrieves context from a deep learning book. "
    "Use the tool to help answer user queries."
)

agent = create_agent(llm, tools, system_prompt=prompt)

创建完成后我们就可以根据书本里的问题对其进行调用了：

In [ ]:
query = (
    "What is the deep learning model used in computer vision tasks? "
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

除了根据提问不断进行信息检索的 Agentic RAG 外，其实我们还可以利用 RAG 技术动态的改造 Agent 的系统提示词，从而让智能体能在发出指令调用工具前有一些具体的参考信息。

比如说根据任务信息在向量数据库中检索比较相近的内容并添加到系统提示词中，这样的话能够帮助智能体更快更好的完成指定的任务。

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """自动根据用户问题检索知识并注入到系统提示"""
    last_query = request.state["messages"][-1].text
    retrieved_docs = vectordb.similarity_search(last_query, k=3)
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
    system_message = (
        "You are a helpful assistant. "
        "Use the following retrieved context to answer the user:\n\n"
        f"{docs_content}"
    )
    print(system_message)
    return system_message

创建完后，可以以中间键的形式添加到 Agent 中：

In [ ]:
from langchain.agents import create_agent 

llm = ChatTongyi(model="qwen-max") 
agent = create_agent(llm, tools=[], middleware=[prompt_with_context])

然后可以正常对其进行调用，这个时候就会把检索到的信息添加到系统提示词后进行回复：

In [ ]:
query = "What is computer vision?"
for step in agent.stream(
  {"messages": [{"role": "user", "content": query}]},
  stream_mode="values",
):
  step["messages"][-1].pretty_print()